# Four-Metric Key 4-gram Calculator
Supplementary Material S2. Identifies move-specific key 4-grams: a 4-gram of the target corpus is key when it meets all four criteria — raw frequency (≥ 3 occurrences in ≥ 3 texts), frequency keyness (G² ≥ 3.84), text dispersion keyness (G² on document counts ≥ 3.84), and mean text frequency keyness (Cohen's *d* > 0). Decisions use unrounded values.

**How to run:** run the cell and answer the prompts (press Enter to accept the default in [brackets]; the defaults reproduce the paper). Then upload **one ZIP of the whole corpus**. The folders (sections and their sub-folders, e.g. moves) are listed with their numbers of .txt files, and target folders are suggested (the sub-folders of the first section, e.g. the three Introduction moves): press Enter to accept, type other numbers (e.g. 6,7,8 or 6-8), or type 'all' for every sub-folder. Each target is analysed separately against the rest of the corpus as reference, and one Excel file per target (`key4grams_<folder>.xlsx`) is downloaded. Hidden files and folders (names starting with '.', e.g. `.DS_Store`, `._*`, and `__MACOSX`) are never read.

**Design (defaults = paper):** statistics on raw surface frequencies; 4-gram windows do not cross `. ? ! ; :`, other punctuation is transparent (*here, we show* = *here we show*); reference = whole corpus **minus** the target folder; G² on the full 2 × 2 table (option: two-term, as in Larsson et al. 2025 keylexbundles); unweighted pooled SD for Cohen's *d*.

**Tokenizer options** (default *yes* = paper): hyphens word-internal (‐ ‑ normalized), apostrophes word-internal (’ ‘ normalized), non-ASCII letters as letters (*NF-κB* = 1 token).

**LBiaP annotation (`counted_lbiap`)** follows the published program of Cortes & Lake (2023) (github.com/lbiap/main): bundles of all lengths up to 9 words that meet the frequency and range criteria are processed longest first and alphabetically within a length; each is counted only where its tokens are unclaimed, and claims them if its recount meets the frequency criterion; frequency and range are then re-checked (`lbiap_retained`). The program's own default thresholds assume a large corpus; here they default to the study's recurrence criterion (f ≥ 3 in ≥ 3 texts) at every length and can be changed. The annotation alters no statistic.

**Output sheets:** `settings`; `extracted 4-grams` (all candidates meeting the frequency criterion, with FK, TDK, MTK, LBiaP counts, `move_specific_key`, and `overlap_family` / `family_lead` for key items — entries merged when the last n−1 words of one are the first n−1 of another; the lead is the most strongly key member); `key 4-grams (Table A)` (the move-specific key 4-grams with f, Txt, FK, TDK, MTK and LBiaP count, ordered by FK and then TDK, as in Appendix Tables A2–A4; `Type` and `Step` are left blank for coding, and `Mv` gives the number of selected target folders in which the 4-gram occurs at least once); `criteria summary` (cumulative counts and each keyness criterion applied alone).

**Summary across targets:** after all targets have been analysed, `summary_tables.xlsx` is downloaded with `Table 1` (tokens, texts, and text length M (SD), median and range for each target and its reference), `Table 3` (candidates retained by each criterion alone and by the four jointly) and `Table A1` (key 4-grams when frequency keyness and text dispersion keyness are both tested at p < .05, .01, .001 and .0001). Folder names are shown as IM1–IM3 through `LABEL` in the code; edit it for other data.

**Next:** code `Type` and `Step` in the `key 4-grams (Table A)` sheets, then run the result-table cell below.

In [ ]:
# Four-Metric Key 4-gram Calculator (paper design: raw statistics + LBiaP counted annotation)
import os,re,math,zipfile,shutil,statistics as st
from collections import Counter
import pandas as pd
try:
    from google.colab import files as _gf
    IN_COLAB=True
except Exception:
    IN_COLAB=False

# ---------------- settings ----------------
# Press Enter to accept the default shown in [brackets]. Invalid answers are asked again.
def ask_int(prompt, default, allowed=None, minimum=None):
    while True:
        a = input(prompt).strip().lower()
        if a in ("", "y", "yes"):
            return default
        try:
            v = int(a)
        except ValueError:
            print(f"  -> please enter a whole number (or press Enter for {default})"); continue
        if allowed is not None and v not in allowed:
            print(f"  -> please enter one of {sorted(allowed)}"); continue
        if minimum is not None and v < minimum:
            print(f"  -> please enter a number >= {minimum}"); continue
        return v
def ask_yn(prompt, default):
    while True:
        a = input(prompt).strip().lower()
        if a == "": return default == "yes"
        if a in ("y", "yes"): return True
        if a in ("n", "no"): return False
        print("  -> please answer yes or no (or press Enter for the default)")

N=ask_int("n-gram length n [4]: ", 4, minimum=2)
KEEP_H=ask_yn("Hyphen word-internal, incl. typographic hyphens \u2010 \u2011 (non-small-cell = 1 token)? (yes/no) [yes]: ", "yes")
KEEP_A=ask_yn("Apostrophe word-internal, incl. curly \u2019 \u2018 (parkinson's = 1 token)? (yes/no) [yes]: ", "yes")
UNI=ask_yn("Treat non-ASCII letters (Greek, accented; NF-\u03baB, TGF-\u03b2) as letters? (yes/no) [yes]: ", "yes")
LL_CELLS=ask_int("Log-likelihood cells? (4 = full table, paper default; 2 = two-term, matches Larsson et al. 2025 keylexbundles) [4]: ", 4, allowed={2, 4})
WEIGHTED=ask_yn("Weighted pooled SD? (yes/no) [no = paper]: ", "no")
EXCL=ask_yn("Exclude target files from reference? (yes/no) [yes = non-overlapping reference, paper]: ", "yes")
CUT_RAW=ask_int("Raw frequency cutoff [3]: ", 3, minimum=1)
CUT_TXT=ask_int("Minimum texts [3]: ", 3, minimum=1)
ANNOT=ask_yn("Add LBiaP 'counted' annotation (interlocking control of Cortes & Lake 2023)? (yes/no) [yes]: ", "yes")
if ANNOT:
    LB_MAXN=ask_int(f"LBiaP: longest bundle length processed (>= {N}) [9]: ", 9, minimum=N)
    LB_FREQ=ask_int("LBiaP: bundle frequency criterion for every length (paper: same as raw cutoff) [3]: ", 3, minimum=1)
    LB_RANGE=ask_int("LBiaP: bundle range criterion, texts (paper: same as minimum texts) [3]: ", 3, minimum=1)
inner=("-" if KEEP_H else "")+("'" if KEEP_A else "")
CH=r"[^\W_]" if UNI else r"[A-Za-z0-9]"          # word characters
WORD=re.compile(rf"{CH}+(?:[{re.escape(inner)}]{CH}+)*") if inner else re.compile(rf"{CH}+")
NORM={}
if KEEP_A: NORM.update({"\u2019":"'","\u2018":"'","\u02bc":"'"})   # typographic apostrophes -> '
if KEEP_H: NORM.update({"\u2010":"-","\u2011":"-"})                # typographic hyphens -> -
NORM=str.maketrans(NORM)
BOUND=re.compile(r"[.!?;:]+")   # sentence-like boundaries; all other punctuation transparent

def is_hidden(name):
    return name.startswith(".") or name == "__MACOSX"      # .DS_Store, ._ files, __MACOSX etc. are never read

def list_txt(root):
    fs=[]
    for dp,dn,fn in os.walk(root):
        dn[:]=sorted(x for x in dn if not is_hidden(x))
        fs+=[os.path.join(dp,f) for f in fn if f.lower().endswith(".txt") and not is_hidden(f)]
    return sorted(fs)

def load_corpus():
    """One ZIP (or folder) holding the whole corpus; target folders are chosen from it afterwards."""
    root="corpus_upload"
    if IN_COLAB:
        print("\nUpload ONE ZIP containing the whole corpus (target and reference folders):")
        up=_gf.upload(); src=list(up)[0]
    else:
        src=input("Path to the corpus ZIP or folder: ").strip()
    if os.path.isdir(src):
        return src
    shutil.rmtree(root, ignore_errors=True); os.makedirs(root)
    with zipfile.ZipFile(src) as z:
        for m in z.infolist():
            if any(is_hidden(p) for p in m.filename.split("/") if p): continue
            z.extract(m, root)
    return root

def parse_numbers(a, n):
    """'2,3,4' or '2-4' or a mix; returns a list of 1-based indices, or None if invalid."""
    out=[]
    for part in a.replace(" ","").split(","):
        if not part: continue
        if "-" in part:
            lo,hi=part.split("-",1)
            if not (lo.isdigit() and hi.isdigit()) or int(lo)>int(hi): return None
            out+=list(range(int(lo),int(hi)+1))
        elif part.isdigit(): out.append(int(part))
        else: return None
    return out if out and all(1<=i<=n for i in out) else None

def choose_targets(root):
    base=root                                   # skip wrapper folders that hold a single sub-folder and no files
    while True:
        subs=[d for d in sorted(os.listdir(base)) if os.path.isdir(os.path.join(base,d)) and not is_hidden(d)]
        files=[f for f in os.listdir(base) if f.lower().endswith(".txt") and not is_hidden(f)]
        if len(subs)==1 and not files: base=os.path.join(base,subs[0])
        else: break
    shown=[]                                    # sections (level 1) and their sub-folders (level 2, e.g. moves)
    for dp,dn,fn in os.walk(base):
        dn[:]=sorted(x for x in dn if not is_hidden(x))
        rel=os.path.relpath(dp,base); depth=0 if rel=="." else rel.count(os.sep)+1
        if depth>2: dn[:]=[]; continue
        n=len(list_txt(dp))
        if n and depth>0: shown.append((dp,n,depth))
    print("\nFolders in the corpus (number of .txt files, sub-folders included):")
    for i,(dp,n,depth) in enumerate(shown,1):
        print(f"  {i:>3}  {'   '*(depth-1)}{os.path.basename(dp)}/  {n}")
    level2=[i for i,x in enumerate(shown,1) if x[2]==2]
    unit=level2 if level2 else [i for i,x in enumerate(shown,1) if x[2]==1]   # the analysable units (e.g. moves)
    first=shown[unit[0]-1][0]; parent=os.path.dirname(first)
    suggested=[i for i in unit if os.path.dirname(shown[i-1][0])==parent] if level2 else unit
    name=lambda i: os.path.relpath(shown[i-1][0],base)
    print(f"\nSuggested targets: {','.join(map(str,suggested))}  ->  " + "; ".join(name(i) for i in suggested))
    print(f"  (each is analysed separately against the rest of the corpus; type 'all' for every {'sub-folder' if level2 else 'folder'} at this level: {len(unit)} analyses)")
    while True:
        a=input("Press Enter to accept the suggestion, or type folder number(s), e.g. 2,3,4 or 2-4, or 'all': ").strip().lower()
        if a in ("", "y", "yes"): idx=suggested
        elif a=="all": idx=unit
        else: idx=parse_numbers(a, len(shown))
        if idx: return base,[shown[i-1][0] for i in idx]
        print(f"  -> please press Enter, type 'all', or enter number(s) from the list (1-{len(shown)})")

def rd(f):
    try: t=open(f,encoding="utf-8").read()
    except UnicodeDecodeError: t=open(f,encoding="utf-8",errors="ignore").read()
    return t.translate(NORM)

def segments(text):
    for s in BOUND.split(text.lower()):
        t=WORD.findall(s)
        if t: yield t

def process(files):
    raw=Counter(); txt=Counter(); pertext=[]; tot=0
    for fp in files:
        fu=Counter(); ntok=0
        for seg in segments(rd(fp)):
            ntok+=len(seg)
            for i in range(len(seg)-N+1): fu[" ".join(seg[i:i+N])]+=1
        tot+=ntok
        for u,c in fu.items(): raw[u]+=c; txt[u]+=1
        pertext.append((ntok,fu))
    return raw,txt,tot,pertext,len(files)

def ll(kT,kR,NT,NR):
    if min(NT,NR)<=0 or kT+kR==0: return 0.0
    Ntot=NT+NR; k=kT+kR
    def cell(O,E): return O*math.log(O/E) if O>0 and E>0 else 0.0
    v=cell(kT,NT*k/Ntot)+cell(kR,NR*k/Ntot)
    if LL_CELLS==4: v+=cell(NT-kT,NT*(Ntot-k)/Ntot)+cell(NR-kR,NR*(Ntot-k)/Ntot)
    v*=2.0
    return -v if kT/NT<kR/NR else v

def cohend(u,tp,rp):
    a=[fu.get(u,0)/nt*1000 for nt,fu in tp if nt>0]; b=[fu.get(u,0)/nt*1000 for nt,fu in rp if nt>0]
    ma,mb=sum(a)/len(a),sum(b)/len(b)
    sa=st.stdev(a) if len(a)>1 else 0.0; sb=st.stdev(b) if len(b)>1 else 0.0
    if WEIGHTED:
        na,nb=len(a),len(b); sd=math.sqrt(((na-1)*sa*sa+(nb-1)*sb*sb)/max(1,na+nb-2))
    else:
        sd=math.sqrt((sa*sa+sb*sb)/2)
    return (ma-mb)/sd if sd>0 else 0.0

def counted_lbiap(files):
    """Interlocking control as in LBiaP (Cortes & Lake 2023; program v0.38, github.com/lbiap/main):
    1) bundles = all n-grams (n = N..LB_MAXN) meeting frequency >= LB_FREQ and range >= LB_RANGE;
    2) lengths are processed longest first, bundles within a length alphabetically;
    3) each bundle is counted, left to right, only where none of its tokens has been claimed;
    4) if that recount still meets LB_FREQ, the bundle claims those token positions,
       so shorter bundles cannot be counted inside it;
    5) frequency and range are re-checked afterwards (reported here as 'lbiap_retained').
    Returns (count per n-gram, texts per n-gram) for length-N bundles."""
    units=[]                                   # (file index, token list)
    for fi,fp in enumerate(files):
        for seg in segments(rd(fp)): units.append((fi,seg))
    lists={}
    for n in range(LB_MAXN,N-1,-1):
        f=Counter(); r={}
        for fi,seg in units:
            for i in range(len(seg)-n+1):
                g=" ".join(seg[i:i+n]); f[g]+=1; r.setdefault(g,set()).add(fi)
        lists[n]=sorted(g for g in f if f[g]>=LB_FREQ and len(r[g])>=LB_RANGE)
    used=[[False]*len(seg) for _,seg in units]
    cnt=Counter(); txt={}
    for n in range(LB_MAXN,N-1,-1):
        for g in lists[n]:
            toks=g.split(" "); hits=[]
            for ui,(fi,seg) in enumerate(units):
                i=0; L=len(seg); u=used[ui]
                while i<=L-n:
                    if seg[i:i+n]==toks and not any(u[i:i+n]):
                        hits.append((ui,i)); i+=n
                    else: i+=1
            if n==N:
                cnt[g]=len(hits); txt[g]=len({units[ui][0] for ui,_ in hits})
            if len(hits)>=LB_FREQ:
                for ui,i in hits:
                    for j in range(i,i+n): used[ui][j]=True
    return cnt,txt

_TG={}
def target_grams(t):
    if t not in _TG: _TG[t]=set(process(list_txt(t))[0])
    return _TG[t]

CRIT=[("p < .05",3.84),("p < .01",6.63),("p < .001",10.83),("p < .0001",15.13)]
# display labels for target folders in the summary tables (edit if needed)
LABEL={"MOVE1":"IM1","MOVE2":"IM2","MOVE3":"IM3"}

def analyse(tname, tfiles, allfiles):
    tset=set(tfiles)
    rfiles=[p for p in allfiles if p not in tset] if EXCL else list(allfiles)
    print(f"\n=== target: {tname} ===")
    print(f"reference: {'whole corpus minus target' if EXCL else 'whole corpus incl. target'} -> {len(rfiles)} files")
    Traw,Ttxt,Ttot,Tp,Tn=process(tfiles); Rraw,Rtxt,Rtot,Rp,Rn=process(rfiles)
    print(f"target tokens={Ttot:,} texts={Tn} | reference tokens={Rtot:,} texts={Rn}")
    cnt,ctxt=counted_lbiap(tfiles) if ANNOT else ({},{})
    rows=[]
    for u,f in Traw.items():
        tx=Ttxt[u]
        if f>=CUT_RAW and tx>=CUT_TXT:
            fk=ll(f,Rraw.get(u,0),Ttot,Rtot); tdk=ll(tx,Rtxt.get(u,0),Tn,Rn)
            d=cohend(u,Tp,Rp)
            rows.append(dict(ngram=u,raw_freq_T=f,texts_T=tx,raw_freq_R=Rraw.get(u,0),texts_R=Rtxt.get(u,0),
                freq_keyness=round(fk,2),disp_keyness=round(tdk,2),mean_keyness=round(d,3),
                counted_lbiap=cnt.get(u,0) if ANNOT else "", counted_texts=ctxt.get(u,0) if ANNOT else "",
                lbiap_retained=(cnt.get(u,0)>=LB_FREQ and ctxt.get(u,0)>=LB_RANGE) if ANNOT else "",
                move_specific_key=(fk>=3.84 and tdk>=3.84 and d>0),
                _fk=fk>=3.84, _tdk=tdk>=3.84, _mtk=d>0, _fkx=fk, _tdkx=tdk))
    COLS=["ngram","raw_freq_T","texts_T","raw_freq_R","texts_R","freq_keyness","disp_keyness","mean_keyness",
          "counted_lbiap","counted_texts","lbiap_retained","move_specific_key","_fk","_tdk","_mtk","_fkx","_tdkx"]
    df=pd.DataFrame(rows,columns=COLS).sort_values(["freq_keyness","disp_keyness"],ascending=False)
    if df.empty: print("  (no 4-gram meets the frequency criterion in this target)")
    # criteria summary on unrounded values: cumulative and each keyness criterion alone (on the recurrence pool)
    fkp,tdp,mtp=df["_fk"],df["_tdk"],df["_mtk"]
    crit=pd.DataFrame([
        ["(a) recurrence pool",len(df)],["cumulative: (a)+FK",int(fkp.sum())],["cumulative: (a)+FK+TDK",int((fkp&tdp).sum())],
        ["cumulative: (a)+FK+TDK+MTK = move-specific key",int(df.move_specific_key.sum())],
        ["alone: FK",int(fkp.sum())],["alone: TDK",int(tdp.sum())],["alone: MTK>0",int(mtp.sum())]],columns=["criterion","n"])
    # Table A: the move-specific key 4-grams, ordered by unrounded FK, then TDK (as in the Appendix)
    tabA=df[df.move_specific_key.astype(bool)].sort_values(["_fkx","_tdkx"],ascending=False,kind="mergesort")
    tabA=tabA[["ngram","raw_freq_T","texts_T","freq_keyness","disp_keyness","mean_keyness"]+(["counted_lbiap"] if ANNOT else [])]
    tabA.columns=["4-gram","f","Txt","FK","TDK","MTK"]+(["Cnt."] if ANNOT else [])
    # columns for coding (fill in, then run the result-table cell): Biber et al. (2004) subtype and step
    tabA["Type"]=""; tabA["Step"]=""
    # Mv = number of selected target folders in which the 4-gram occurs at least once
    tabA["Mv"]=[sum(g in target_grams(t) for t in TARGETS) for g in tabA["4-gram"]]
    # Table A1: key 4-grams when FK and TDK are both tested at stricter critical values (recurrence and MTK unchanged)
    sens=[(lab,c,int(((df["_fkx"]>=c)&(df["_tdkx"]>=c)&df["_mtk"].astype(bool)).sum()) if len(df) else 0) for lab,c in CRIT]
    summ=dict(name=tname,Ttot=Ttot,Tn=Tn,Rtot=Rtot,Rn=Rn,Tlen=[x[0] for x in Tp],Rlen=[x[0] for x in Rp],
              pool=len(df),fk=int(fkp.sum()),tdk=int(tdp.sum()),mtk=int(mtp.sum()),key=int(df.move_specific_key.sum()),sens=sens)
    df=df.drop(columns=["_fk","_tdk","_mtk","_fkx","_tdkx"])
    # overlap families among move-specific key items: merge entries whose last n-1 words are the first n-1 of another
    key_items=list(df[df.move_specific_key.astype(bool)].sort_values(["freq_keyness","disp_keyness"],ascending=False).ngram)
    par={g:g for g in key_items}
    def _find(x):
        while par[x]!=x: par[x]=par[par[x]]; x=par[x]
        return x
    heads={}
    for g in key_items: heads.setdefault(tuple(g.split()[:-1]),[]).append(g)
    for g in key_items:
        for h in heads.get(tuple(g.split()[1:]),[]):
            if h!=g: par[_find(g)]=_find(h)
    lead={}
    for g in key_items: lead.setdefault(_find(g),g)          # first = most strongly key member
    fam_id={r:i+1 for i,r in enumerate(lead)}
    df["overlap_family"]=[fam_id[_find(g)] if g in par else "" for g in df.ngram]
    df["family_lead"]=[lead[_find(g)] if g in par else "" for g in df.ngram]
    crit.loc[len(crit)]=["overlap families among move-specific key items",len(lead)]
    print("candidates:",len(df),"| move-specific key:",int(df['move_specific_key'].sum()))
    out=f"key4grams_{re.sub(r'[^A-Za-z0-9]+','_',tname).strip('_')}.xlsx"
    with pd.ExcelWriter(out,engine="openpyxl") as w:
        pd.DataFrame([["design","raw surface frequencies; counted_lbiap = annotation by LBiaP interlocking control (Cortes & Lake 2023)"],
            ["boundaries",". ? ! ; : block windows; other punctuation transparent"],
            ["tokenizer",f"hyphen internal={KEEP_H} (\u2010\u2011 normalized), apostrophe internal={KEEP_A} (\u2019\u2018 normalized), non-ASCII letters={UNI}"],
            ["rounding","FK/TDK shown to 2 dp, MTK to 3 dp; move-specific key status is decided on unrounded values"],
            ["LL cells",LL_CELLS],["pooled SD","weighted" if WEIGHTED else "unweighted sqrt((s1^2+s2^2)/2)"],
            ["reference","target excluded" if EXCL else "target included"],
            ["cutoffs",f"raw>={CUT_RAW}, texts>={CUT_TXT}"],
            ["LBiaP",(f"lengths {LB_MAXN}..{N}, longest first, alphabetical within length; bundle criterion f>={LB_FREQ} in >={LB_RANGE} texts" if ANNOT else "off")],
            ["target",f"{tname}: {Ttot} tokens / {Tn} texts"],
            ["Table A","Type and Step are left blank for coding (Biber et al. 2004 subtype; Step 1, Step 2, ... or NS); Mv = number of selected targets in which the 4-gram occurs"],["reference",f"{Rtot} tokens / {Rn} texts"]]
            ).to_excel(w,sheet_name="settings",index=False,header=False)
        df.to_excel(w,sheet_name="extracted 4-grams",index=False)
        tabA.to_excel(w,sheet_name="key 4-grams (Table A)",index=False)
        crit.to_excel(w,sheet_name="criteria summary",index=False)
    if IN_COLAB: _gf.download(out)
    print("wrote",out)
    return summ

root=load_corpus()
allfiles=list_txt(root)
print(f"corpus: {len(allfiles)} .txt files (hidden files ignored)")
base,targets=choose_targets(root)
TARGETS=list(targets)
SUMM=[]
for t in targets:
    SUMM.append(analyse(os.path.relpath(t,base), list_txt(t), allfiles))

# ---------------- summary tables across all selected targets (Table 1, Table 3, Table A1) ----------------
def write_summary(S, out="summary_tables.xlsx"):
    from openpyxl import Workbook
    from openpyxl.styles import Font, Alignment, Border, Side
    F="Times New Roman"; thin=Side(style="thin")
    lab=lambda x: LABEL.get(os.path.basename(x["name"]), x["name"])
    def desc(v):
        v=sorted(v); m=st.mean(v); sd=st.stdev(v) if len(v)>1 else 0.0
        return f"{m:.1f} ({sd:.1f})", f"{st.median(v):.1f}", f"{v[0]:,}–{v[-1]:,}"
    t1=[["Move","Corpus","Tokens","Texts","Length M (SD)","Mdn","Range"]]
    for x in S:
        t1.append([lab(x),"Target",f"{x['Ttot']:,}",f"{x['Tn']:,}",*desc(x["Tlen"])])
        t1.append(["","Reference",f"{x['Rtot']:,}",f"{x['Rn']:,}",*desc(x["Rlen"])])
    t3=[["Criterion"]+[lab(x) for x in S],
        ["Raw frequency ≥ %d in ≥ %d texts (candidates)"%(CUT_RAW,CUT_TXT)]+[x["pool"] for x in S],
        ["Frequency keyness alone"]+[x["fk"] for x in S],
        ["Text dispersion keyness alone"]+[x["tdk"] for x in S],
        ["Mean text frequency keyness > 0 alone"]+[x["mtk"] for x in S],
        ["All four criteria (key 4-grams)"]+[x["key"] for x in S]]
    ta1=[["Significance level","Critical G² (FK and TDK)"]+[lab(x) for x in S]+["Total"]]
    for i,(p,c) in enumerate(CRIT):
        v=[x["sens"][i][2] for x in S]
        ta1.append([p+(" (adopted)" if i==0 else ""),f"{c:.2f}"]+v+[sum(v)])
    tabs=[("Table 1","Target and Reference Corpora",t1,"Note. Text length in tokens; Mdn = median."),
          ("Table 3","Candidate 4-grams Retained by Each Criterion Alone and by the Four Criteria Jointly",t3,
           "Note. The keyness criteria were applied to the candidates meeting the frequency criterion and decided on unrounded values."),
          ("Table A1","Number of Key 4-grams When Frequency Keyness and Text Dispersion Keyness Are Tested at Four Significance Levels",ta1,
           "Note. A 4-gram is counted when both frequency keyness and text dispersion keyness reach the critical value; the recurrence and mean text frequency keyness criteria are unchanged.")]
    wb=Workbook(); wb.remove(wb.active)
    for name,cap,rows,note in tabs:
        ws=wb.create_sheet(name)
        ws["A1"]=name; ws["A1"].font=Font(name=F,bold=True)
        ws["A2"]=cap; ws["A2"].font=Font(name=F,italic=True)
        for i,r in enumerate(rows,4):
            for j,v in enumerate(r,1):
                c=ws.cell(i,j,v); c.font=Font(name=F)
                c.alignment=Alignment(horizontal="left" if j==1 or (name=="Table 1" and j==2) else "center")
                if i==4: c.border=Border(top=thin,bottom=thin)
                elif i==3+len(rows): c.border=Border(bottom=thin)
        ws.cell(5+len(rows),1,note).font=Font(name=F)
        ws.column_dimensions["A"].width=44 if name!="Table 1" else 10
        for col in "BCDEFG": ws.column_dimensions[col].width=16
    wb.save(out); print("wrote",out)
    if IN_COLAB: _gf.download(out)
write_summary(SUMM)


# Result tables (Table 4, Table 5, Tables A2–A4)
Run this cell after the key 4-grams have been coded. Upload the coded workbook(s): either the `key4grams_<folder>.xlsx` files from the cell above, or any workbook whose sheets have the columns `4-gram`, `Type` and `Step` (one sheet per move).

**Coding:** `Type` = subtype code of Biber, Conrad & Cortes (2004, p. 381, Table 2): 1a–1g (verb phrase fragments), 2a–2e (dependent clause fragments), 3a–3e (noun phrase and prepositional phrase fragments). `Step` = `Step 1`, `Step 2`, … (step numbers restart within each move) or `NS` (not step-signalling). Invalid codes are listed when the files are read.

**Output:** `result_tables.xlsx` with `Table 4` (all 19 subtypes in the order and wording of Biber et al. 2004, Table 2, with n (%) per move), `Table 5` (steps × structural types per move, with the four most strongly key 4-grams of each row), and one `Table A` sheet per move. Move and step names can be edited in `MOVE_CODE`, `MOVE_TITLE` and `STEP_TITLE` at the top of the cell.

In [ ]:
# Result tables: structural types (Biber et al., 2004, Table 2) and steps of the key 4-grams
# Input : one or more .xlsx files with coded key 4-grams. Every sheet that has the columns
#         "4-gram", "Type" and "Step" is read as one move (e.g. the "key 4-grams (Table A)" sheet
#         written by the calculator above, after Type and Step have been filled in).
#         Type = Biber et al. (2004) subtype code (1a-1g, 2a-2e, 3a-3e)
#         Step = "Step 1", "Step 2", ... or "NS" (not step-signalling)
# Output: result_tables.xlsx with Table 4 (structural types by move), Table 5 (steps x types)
#         and one Table A sheet per move.
import os, re, glob
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side
try:
    from google.colab import files as _gf
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ---- display names (edit if needed) -------------------------------------------------------
# raw name of a move (sheet name, or the folder name in key4grams_<folder>.xlsx) -> label
MOVE_CODE = {"MOVE1": "IM1", "MOVE2": "IM2", "MOVE3": "IM3",
             "Table A2": "IM1", "Table A3": "IM2", "Table A4": "IM3"}
MOVE_TITLE = {"IM1": "Presenting background information",
              "IM2": "Reviewing related research",
              "IM3": "Presenting new research"}
STEP_TITLE = {"IM1": {"Step 1": "Established knowledge", "Step 2": "Main research problems"},
              "IM2": {"Step 1": "Previous research", "Step 2": "Limitations of previous research"},
              "IM3": {"Step 1": "Research purpose", "Step 2": "Main research procedure"}}
N_REP = 4          # representative 4-grams per row of Table 5 (highest frequency keyness first)

# ---- Biber, Conrad & Cortes (2004, p. 381, Table 2) ------------------------------------------
BIBER = [
 ("1", "Lexical bundles that incorporate verb phrase fragments", [
   ("1a", "(connector +) 1st/2nd person pronoun + VP fragment"),
   ("1b", "(connector +) 3rd person pronoun + VP fragment"),
   ("1c", "Discourse marker + VP fragment"),
   ("1d", "Verb phrase (with non-passive verb)"),
   ("1e", "Verb phrase with passive verb"),
   ("1f", "yes–no question fragments"),
   ("1g", "WH-question fragments")]),
 ("2", "Lexical bundles that incorporate dependent clause fragments", [
   ("2a", "1st/2nd person pronoun + dependent clause fragment"),
   ("2b", "WH-clause fragments"),
   ("2c", "If-clause fragments"),
   ("2d", "(verb/adjective+) to-clause fragment"),
   ("2e", "That-clause fragments")]),
 ("3", "Lexical bundles that incorporate noun phrase and prepositional phrase fragments", [
   ("3a", "(connector +) Noun phrase with of-phrase fragment"),
   ("3b", "Noun phrase with other post-modifier fragment"),
   ("3c", "Other noun phrase expressions"),
   ("3d", "Prepositional phrase expressions"),
   ("3e", "Comparative expressions")])]
CODES = [c for _, _, s in BIBER for c, _ in s]

def norm_step(s):
    s = str(s).strip()
    m = re.fullmatch(r"(?i)step\s*(\d+)", s)
    if m: return f"Step {m.group(1)}"
    if s.upper() in ("NS", "NOT STEP-SIGNALLING", "NOT STEP-SIGNALING"): return "NS"
    return s

def read_coded(paths):
    moves = []
    for p in sorted(paths):
        stem = re.sub(r"^key4grams_", "", os.path.splitext(os.path.basename(p))[0])
        for sh, df in pd.read_excel(p, sheet_name=None, header=None).items():
            # find the header row (the one containing "4-gram")
            hr = next((i for i in range(min(10, len(df))) if "4-gram" in [str(x).strip() for x in df.iloc[i]]), None)
            if hr is None: continue
            d = df.iloc[hr + 1:].copy(); d.columns = [str(x).strip() for x in df.iloc[hr]]
            if not {"4-gram", "Type", "Step"} <= set(d.columns): continue
            d = d[d["4-gram"].notna() & (d["4-gram"].astype(str).str.strip() != "")]
            raw = sh if sh != "key 4-grams (Table A)" else stem
            key = raw.split("_")[-1] if raw not in MOVE_CODE else raw
            name = MOVE_CODE.get(raw, MOVE_CODE.get(key, raw))
            d["Type"] = d["Type"].astype(str).str.strip().str.lower()
            d["Step"] = d["Step"].map(norm_step)
            bad_t = d[~d["Type"].isin(CODES)]; bad_s = d[~d["Step"].str.fullmatch(r"Step \d+|NS")]
            for _, r in bad_t.iterrows(): print(f"  [{name}] invalid Type '{r['Type']}' for: {r['4-gram']}")
            for _, r in bad_s.iterrows(): print(f"  [{name}] invalid Step '{r['Step']}' for: {r['4-gram']}")
            moves.append((name, d.reset_index(drop=True)))
            print(f"read {os.path.basename(p)} / {sh} -> {name}: {len(d)} key 4-grams")
    return moves

def pct(n, N): return f"{n} ({100 * n / N:.1f})" if N else "0"

def table4(moves):
    rows = []
    for t, lab, subs in BIBER:
        cnt = [int(d["Type"].isin([c for c, _ in subs]).sum()) for _, d in moves]
        rows.append([f"{t}. {lab}"] + [pct(c, len(d)) for c, (_, d) in zip(cnt, moves)] + [sum(cnt)])
        for c, l in subs:
            cc = [int((d["Type"] == c).sum()) for _, d in moves]
            rows.append([f"{c}. {l}"] + cc + [sum(cc)])
    rows.append(["Total"] + [f"{len(d)} (100)" for _, d in moves] + [sum(len(d) for _, d in moves)])
    return pd.DataFrame(rows, columns=["Structural type"] + [f"{m} n (%)" for m, _ in moves] + ["Total"])

def table5(moves):
    rows = []
    def sortkey(s): return (1, 0) if s == "NS" else (0, int(s.split()[1]))
    for m, d in moves:
        rows.append([f"{m} {MOVE_TITLE.get(m, '')}".strip(), "", "", "", "", ""])
        for s in sorted(d["Step"].unique(), key=sortkey):
            g = d[d["Step"] == s]
            if "FK" in g.columns: g = g.sort_values("FK", ascending=False, kind="mergesort")
            lab = "Not step-signalling" if s == "NS" else f"{s} {STEP_TITLE.get(m, {}).get(s, '')}".strip()
            bt = g["Type"].str[0]
            rep = ", ".join(x.replace(" †", "") for x in g["4-gram"].astype(str).head(N_REP))
            rows.append([" " + lab, len(g), int((bt == "1").sum()), int((bt == "2").sum()), int((bt == "3").sum()), rep])
    allt = pd.concat([d for _, d in moves])["Type"].str[0]
    rows.append(["Total", sum(len(d) for _, d in moves), int((allt == "1").sum()), int((allt == "2").sum()), int((allt == "3").sum()), ""])
    return pd.DataFrame(rows, columns=["Move and step", "n", "Type 1", "Type 2", "Type 3", "Representative key 4-grams"])

def write(moves, out="result_tables.xlsx"):
    F = "Times New Roman"; thin = Side(style="thin")
    wb = Workbook(); wb.remove(wb.active)
    def put(title, caption, df, note, italic_col=None, widths=None):
        ws = wb.create_sheet(title)
        ws["A1"] = title; ws["A1"].font = Font(name=F, bold=True)
        ws["A2"] = caption; ws["A2"].font = Font(name=F, italic=True)
        for j, h in enumerate(df.columns, 1):
            c = ws.cell(4, j, h); c.font = Font(name=F); c.border = Border(top=thin, bottom=thin)
            c.alignment = Alignment(horizontal="left" if j == 1 else "center", wrap_text=True)
        for i, r in enumerate(df.itertuples(index=False), 5):
            for j, v in enumerate(r, 1):
                c = ws.cell(i, j, v)
                head = isinstance(r[0], str) and (re.match(r"^\d\. ", r[0]) or r[0] == "Total")
                c.font = Font(name=F, italic=(italic_col is not None and j in italic_col), bold=False)
                c.alignment = Alignment(horizontal="left" if j == 1 or (italic_col and j in italic_col) else "center",
                                        indent=(1 if j == 1 and isinstance(r[0], str) and re.match(r"^\d[a-g]\. ", r[0]) else 0))
        last = 4 + len(df)
        for j in range(1, len(df.columns) + 1): ws.cell(last, j).border = Border(bottom=thin)
        ws.cell(last + 2, 1, note).font = Font(name=F)
        for k, w in enumerate(widths or [], 1): ws.column_dimensions[chr(64 + k)].width = w
    put("Table 4", "Structural Types of Key 4-grams by Move (Biber et al., 2004)", table4(moves),
        "Note. Types and subtypes follow Biber et al. (2004, p. 381, Table 2); subtype 1b includes noun-phrase subjects (studies have shown that).",
        widths=[70] + [14] * len(moves) + [8])
    put("Table 5", "Steps Signalled by Key 4-grams and Their Structural Types", table5(moves),
        "Note. Step numbers restart within each move. Type 1 = verb phrase, Type 2 = dependent clause, Type 3 = noun- and prepositional-phrase fragments (Biber et al., 2004).",
        italic_col={6}, widths=[40, 6, 8, 8, 8, 70])
    for m, d in moves:
        cols = [c for c in ["4-gram", "f", "Txt", "FK", "TDK", "MTK", "Cnt.", "Type", "Step", "Mv"] if c in d.columns]
        put(f"Table A {m}"[:31], f"Key 4-grams in {m}" + (f" ({MOVE_TITLE[m].title()})" if m in MOVE_TITLE else "") + f", n = {len(d)}",
            d[cols], "Note. f = raw frequency; Txt = texts; FK = frequency keyness; TDK = text dispersion keyness; "
            "MTK = mean text frequency keyness; Cnt. = occurrences counted under LBiaP interlocking control; "
            "† = technical 4-gram; Step: NS = not step-signalling; Mv = number of moves (selected targets) in which the item occurs at least once.",
            italic_col={1}, widths=[36] + [8] * (len(cols) - 1))
    wb.save(out); print("wrote", out)
    if IN_COLAB: _gf.download(out)
    return out

if __name__ == "__main__":
    if IN_COLAB:
        print("Upload the coded workbook(s) (Type and Step filled in):")
        up = _gf.upload(); paths = list(up)
    else:
        import sys; paths = sys.argv[1:] or glob.glob("key4grams_*.xlsx")
    moves = read_coded(paths)
    if not moves: raise SystemExit("No sheet with the columns 4-gram, Type and Step was found.")
    write(moves)
